# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR²) Exploration with `mlcroissant`

This notebook provides a template for loading, exploring, and processing the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. We will:

1. Load the dataset via its [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json).
2. Review available record sets, fields, and their `@id`s (as per Croissant best practices).
3. Extract and process data by `@id`.
4. Perform exploratory analysis and simple visualization.

### Dataset Source
The dataset schema is published at: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. 

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant metadata URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print a summary
metadata_json = dataset.metadata.to_json()
print(f"Dataset Name: {metadata_json['name']}")
print(f"Description: {metadata_json['description']}")
print(f"Version: {metadata_json.get('version', '--')}")
print(f"Identifier: {metadata_json.get('identifier', '--')}")


## 2. Data Overview
Review the available record sets, their fields, and Croissant `@id`s. Referencing by `@id` ensures clarity and reproducibility.

In [ ]:
# List available record sets and their fields by @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):  # single field case
        fields = [fields]
    print("  Fields:")
    for f in fields:
        if isinstance(f, dict):
            print(f"    - {f.get('@id', '--')} (name: {f.get('name', '--')})")
        else:
            print(f"    - {f}")
    print("")

## 3. Data Extraction

Extract all record sets by their `@id` and load them into Pandas DataFrames for further analysis.

The extraction process below uses only the Croissant `@id`s for all references (record sets, fields, etc.).

In [ ]:
# Prepare the list of record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = dict()
# Load each record set using its @id and convert to DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Loaded {len(records)} records for record set: {record_set_id}")
    else:
        print(f"No records found for record set: {record_set_id}")

# Display fields of the first non-empty record set
first_rs = None
for k, v in dataframes.items():
    if not v.empty:
        first_rs = k
        break
if first_rs:
    print(f"\nColumns in record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())
else:
    print("No non-empty record sets found.")

## 4. Exploratory Data Analysis (EDA)

Now that the data is loaded, let's apply some classic EDA steps: filter and normalize a numeric field, and group by a key attribute.

We'll use `@id` to reference fields. Please identify available numeric and group fields from the data overview above.

In [ ]:
# Example: filter and normalize a numeric field, group by another
# Make sure to adjust `numeric_field_id` and `group_field_id` below to actual values from your dataset
# For the FAIR² dataset, let's use common clinical fields (e.g., 'age', 'interval_between_diagnoses', etc.)

# Identify a suitable record set and fields (edit these IDs as per overview above if needed)
record_set_id = first_rs  # use the first non-empty record set

numeric_field_id = None
# Simple heuristic to find a likely numeric field
for col in dataframes[record_set_id].columns:
    if 'age' in col.lower() or 'interval' in col.lower() or 'year' in col.lower():
        numeric_field_id = col
        break
if not numeric_field_id:
    # fallback to the first column
    numeric_field_id = dataframes[record_set_id].columns[0]

group_field_id = None
for col in dataframes[record_set_id].columns:
    if 'sex' in col.lower() or 'gender' in col.lower() or 'msi' in col.lower() or 'location' in col.lower():
        group_field_id = col
        break

df = dataframes[record_set_id].copy()

# Convert to numeric, errors ignored
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Remove extreme values/outliers (optional): filter over a threshold
threshold = df[numeric_field_id].mean() if pd.notnull(df[numeric_field_id].mean()) else 10
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records from '{record_set_id}' with field '{numeric_field_id}' > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
if not filtered_df.empty:
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / 
        filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean '{numeric_field_id}' by '{group_field_id}':")
        display(grouped_df.head())
else:
    print("No records found above threshold for this numeric field.")

## 5. Visualization

Visualize distributions or relationships between fields (using only `@id` as axis labels where possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: histogram and boxplot of the numeric field
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
df[numeric_field_id].dropna().plot.hist(bins=10, alpha=0.7)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)

plt.subplot(1, 2, 2)
sns.boxplot(y=df[numeric_field_id])
plt.title(f"Boxplot of {numeric_field_id}")
plt.ylabel(numeric_field_id)
plt.tight_layout()
plt.show()

# If grouping field is available, plot means
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion

- We successfully loaded and explored the FAIR² dataset via its Croissant schema using the `mlcroissant` library.
- All entities and fields were referenced strictly by `@id` for reproducible, standards-based data access.
- Simple EDA and visualizations on numeric and categorical fields demonstrate the clinical dataset's structure, supporting further downstream analysis or modeling.

You can now proceed with deeper statistical analysis, visualization, or machine learning tasks using these loaded DataFrames.